# Home Credit Default Risk — C: Modeling, Cross-Validation and Tuning

This notebook contains the modeling work for Part C. 
It reuses the fixed data split provided by Part A and will later incorporate 
the V1 feature decisions provided by Part B.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

# Locate the project root
ROOT = Path.cwd()

if not (ROOT / "src").exists():
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Project root could not be located.")

# Make shared project modules importable
sys.path.insert(0, str(ROOT / "src"))

# Reuse Part A's shared data pipeline
from data_pipeline import load_main_tables, fixed_split
from baseline import build_baseline

# Load the official main tables
train, test, dictionary = load_main_tables(ROOT)

# Reuse Part A's fixed 80/20 split
fit_idx, val_idx, splits = fixed_split(ROOT, train)

print("Project root:", ROOT)
print("Training table:", train.shape)
print("Test table:", test.shape)
print("Training split:", len(fit_idx))
print("Validation split:", len(val_idx))

Project root: e:\MSDM5054\msdm5054-home-credit
Training table: (307511, 122)
Test table: (48744, 121)
Training split: 246008
Validation split: 61503


## 2. V0 Logistic Regression Baseline

We first reproduce the V0 baseline provided by Part A using the same fixed
training/validation split. This provides a common reference point for all
subsequent model comparisons.

In [2]:
from sklearn.metrics import roc_auc_score

# Build Part A's V0 baseline pipeline
v0_baseline = build_baseline(seed=42)

# Training data and labels
X_fit = train.iloc[fit_idx]
y_fit = train["TARGET"].iloc[fit_idx]

# Validation data and labels
X_val = train.iloc[val_idx]
y_val = train["TARGET"].iloc[val_idx]

# Train the baseline model
v0_baseline.fit(X_fit, y_fit)

# Predict default probabilities on the validation set
v0_val_prob = v0_baseline.predict_proba(X_val)[:, 1]

# Calculate validation ROC-AUC
v0_auc = roc_auc_score(y_val, v0_val_prob)

print(f"V0 Logistic Regression Validation ROC-AUC: {v0_auc:.6f}")

V0 Logistic Regression Validation ROC-AUC: 0.751239


## 3. V0 LightGBM Baseline

We now replace the Logistic Regression classifier with LightGBM while keeping
the same V0 data, fixed train/validation split, and preprocessing framework.
This allows a fair comparison between a linear baseline and a nonlinear
tree-based model.

In [3]:
import lightgbm as lgb

print("LightGBM version:", lgb.__version__)

LightGBM version: 4.7.0


In [4]:
from data_pipeline import HomeCreditPreprocessor
import inspect

print(inspect.signature(HomeCreditPreprocessor))
print(v0_baseline)

(missing_threshold=0.8)
Pipeline(steps=[('preprocess', HomeCreditPreprocessor()),
                ('classifier',
                 LogisticRegression(C=1, max_iter=2000, random_state=42))])


In [5]:
from sklearn.pipeline import Pipeline
from sklearn.base import clone
from lightgbm import LGBMClassifier

# Reuse Part A's V0 preprocessing,
# but replace Logistic Regression with LightGBM
v0_lgbm = Pipeline([
    (
        "preprocess",
        clone(v0_baseline.named_steps["preprocess"])
    ),
    (
        "classifier",
        LGBMClassifier(
            objective="binary",
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            random_state=42,
            n_jobs=-1,
            verbosity=-1
        )
    )
])

# Train on exactly the same training split
v0_lgbm.fit(X_fit, y_fit)

# Predict probabilities on exactly the same validation split
v0_lgbm_val_prob = v0_lgbm.predict_proba(X_val)[:, 1]

# Calculate ROC-AUC
v0_lgbm_auc = roc_auc_score(y_val, v0_lgbm_val_prob)

print(f"V0 Logistic Regression ROC-AUC: {v0_auc:.6f}")
print(f"V0 LightGBM ROC-AUC:            {v0_lgbm_auc:.6f}")
print(f"AUC improvement:                {v0_lgbm_auc - v0_auc:+.6f}")

V0 Logistic Regression ROC-AUC: 0.751239
V0 LightGBM ROC-AUC:            0.759364
AUC improvement:                +0.008124


## 4. Model Comparison

The following table records model performance under the same fixed
train/validation split. This provides a controlled comparison between
different model classes.

In [6]:
results = pd.DataFrame([
    {
        "version": "V0",
        "model": "Logistic Regression",
        "features": "A V0 features",
        "validation": "Fixed 80/20",
        "roc_auc": v0_auc
    },
    {
        "version": "V0",
        "model": "LightGBM",
        "features": "A V0 features",
        "validation": "Fixed 80/20",
        "roc_auc": v0_lgbm_auc
    }
])

results

,version,model,features,validation,roc_auc
0,V0,Logistic Regression,A V0 features,Fixed 80/20,0.751239
1,V0,LightGBM,A V0 features,Fixed 80/20,0.759364


## 5. V1 Feature Handoff from Part B

Part B provides a column-level decision table based on EDA, IV, collinearity,
outlier analysis, and tree-based cross-checking. We first validate the handoff
file before implementing the V1 preprocessing pipeline.

In [7]:
# Load Part B's machine-readable feature decisions
decision_path = ROOT / "results" / "eda" / "column_decisions.csv"

column_decisions = pd.read_csv(decision_path)

print("Decision file shape:", column_decisions.shape)
print("\nColumns:")
print(column_decisions.columns.tolist())

print("\nDecision counts:")
print(column_decisions["decision"].value_counts(dropna=False))

print("\nTransform counts:")
print(column_decisions["transform"].fillna("none").value_counts(dropna=False))

column_decisions.head(10)

Decision file shape: (121, 6)

Columns:
['feature', 'decision', 'transform', 'reason', 'iv', 'type']

Decision counts:
decision
KEEP    65
DROP    55
ADD      1
Name: count, dtype: int64

Transform counts:
transform
none                                                 87
missing indicator + bin                              17
rare-merge + label encode                            10
rare-merge + target encode                            2
cap at P99.9                                          2
AMT_ANNUITY / AMT_CREDIT; zero denominator -> NaN     1
sentinel 365243 -> NaN                                1
log1p; cap at P99.9                                   1
Name: count, dtype: int64


,feature,decision,transform,reason,iv,type
0,ANNUITY_TO_CREDIT,ADD,AMT_ANNUITY / AMT_CREDIT; zero denominator -> NaN,only derived ratio beating both parents (IV 0....,0.0915,derived
1,REGION_RATING_CLIENT,DROP,NaN,collinear with REGION_RATING_CLIENT_W_CITY (|r...,0.0482,ordinal
2,AMT_CREDIT,DROP,NaN,collinear with AMT_GOODS_PRICE (|r| > 0.9); it...,0.0448,continuous
3,FLOORSMAX_MEDI,DROP,NaN,collinear with FLOORSMAX_AVG (|r| > 0.9),0.0379,continuous
4,FLOORSMAX_MODE,DROP,NaN,collinear with FLOORSMAX_AVG (|r| > 0.9),0.0374,continuous
5,LIVINGAREA_AVG,DROP,NaN,collinear with TOTALAREA_MODE (|r| > 0.9),0.0328,continuous
6,LIVINGAREA_MEDI,DROP,NaN,collinear with TOTALAREA_MODE (|r| > 0.9),0.0325,continuous
7,APARTMENTS_AVG,DROP,NaN,collinear with TOTALAREA_MODE (|r| > 0.9),0.0321,continuous
8,LIVINGAREA_MODE,DROP,NaN,collinear with TOTALAREA_MODE (|r| > 0.9),0.0320,continuous
9,APARTMENTS_MEDI,DROP,NaN,collinear with TOTALAREA_MODE (|r| > 0.9),0.0317,continuous


In [8]:
keep_features = column_decisions.loc[
    column_decisions["decision"] == "KEEP",
    "feature"
].tolist()

add_features = column_decisions.loc[
    column_decisions["decision"] == "ADD",
    "feature"
].tolist()

drop_features = column_decisions.loc[
    column_decisions["decision"] == "DROP",
    "feature"
].tolist()

print("KEEP features:", len(keep_features))
print("ADD features:", len(add_features))
print("DROP features:", len(drop_features))
print("Total V1 model features:", len(keep_features) + len(add_features))

print("\nAdded feature:")
print(add_features)

KEEP features: 65
ADD features: 1
DROP features: 55
Total V1 model features: 66

Added feature:
['ANNUITY_TO_CREDIT']


## 6. V1 Transformation Groups

The V1 preprocessing rules are derived from Part B's machine-readable
feature decisions and univariate flags. All transformations that require
estimated parameters will be fitted using training data only.

In [9]:
import json

# Load Part B's univariate flags
flags_path = ROOT / "results" / "eda" / "univariate_flags.json"

with open(flags_path, "r", encoding="utf-8") as f:
    flags = json.load(f)

keep_set = set(keep_features)

# Transformation groups among the 65 retained original features
log_features = sorted(
    set(flags["log_transform"]) & keep_set
)

high_card_features = sorted(
    set(flags["high_cardinality"]) & keep_set
)

sparse_cont_features = sorted(
    set(flags["sparse_continuous"]) & keep_set
)

sparse_cat_features = sorted(
    set(flags["sparse_categorical"]) & keep_set
)

cap_features = sorted(
    set(flags["extreme_outlier"]) & keep_set
)

# Read label-encoded variables directly from B's final decision table
label_features = column_decisions.loc[
    (column_decisions["decision"] == "KEEP")
    & column_decisions["transform"].fillna("").str.contains("label encode"),
    "feature"
].tolist()

print("Log1p features:", len(log_features), log_features)
print()
print("Target-encoded features:", len(high_card_features), high_card_features)
print()
print("Label-encoded features:", len(label_features), label_features)
print()
print("Sparse continuous features:", len(sparse_cont_features))
print(sparse_cont_features)
print()
print("Sparse categorical features:", len(sparse_cat_features))
print(sparse_cat_features)
print()
print("P99.9 capped features:", len(cap_features), cap_features)

Log1p features: 1 ['AMT_INCOME_TOTAL']

Target-encoded features: 2 ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

Label-encoded features: 10 ['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START', 'FLAG_OWN_REALTY']

Sparse continuous features: 13
['BASEMENTAREA_AVG', 'COMMONAREA_AVG', 'ELEVATORS_AVG', 'ENTRANCES_AVG', 'EXT_SOURCE_1', 'FLOORSMAX_AVG', 'FLOORSMIN_MEDI', 'LANDAREA_AVG', 'NONLIVINGAPARTMENTS_MEDI', 'NONLIVINGAREA_AVG', 'OWN_CAR_AGE', 'TOTALAREA_MODE', 'YEARS_BEGINEXPLUATATION_MEDI']

Sparse categorical features: 4
['EMERGENCYSTATE_MODE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']

P99.9 capped features: 3 ['AMT_INCOME_TOTAL', 'AMT_REQ_CREDIT_BUREAU_QRT', 'OBS_30_CNT_SOCIAL_CIRCLE']


## 7. Build the V1 Raw Feature Matrix

We construct the 66 V1 model features defined by Part B while preserving
Part A's fixed train/validation split. Transformations and encodings will be
fitted later using training data only to avoid data leakage.

In [11]:
def build_v1_raw_features(df):
    """
    Construct the raw V1 feature matrix from Part B's feature decisions.
    No learned preprocessing is performed here.
    """
    X = df.copy()

    # Derived feature from Part B
    denominator = X["AMT_CREDIT"].replace(0, np.nan)
    X["ANNUITY_TO_CREDIT"] = X["AMT_ANNUITY"] / denominator

    # Keep exactly the 65 retained features + 1 derived feature
    v1_features = keep_features + add_features
    X = X[v1_features].copy()

    return X


# Build V1 feature matrices
X_v1 = build_v1_raw_features(train)
X_v1_test = build_v1_raw_features(test)

# Reuse Part A's fixed split
X_v1_fit = X_v1.iloc[fit_idx].copy()
X_v1_val = X_v1.iloc[val_idx].copy()

y_v1_fit = train["TARGET"].iloc[fit_idx].copy()
y_v1_val = train["TARGET"].iloc[val_idx].copy()

print("Full V1 matrix:", X_v1.shape)
print("V1 training split:", X_v1_fit.shape)
print("V1 validation split:", X_v1_val.shape)
print("V1 test matrix:", X_v1_test.shape)

print("\nANNUITY_TO_CREDIT included:",
      "ANNUITY_TO_CREDIT" in X_v1.columns)

print("AMT_CREDIT used as model feature:",
      "AMT_CREDIT" in X_v1.columns)

print("TARGET accidentally included:",
      "TARGET" in X_v1.columns)

print("SK_ID_CURR accidentally included:",
      "SK_ID_CURR" in X_v1.columns)

assert X_v1.shape[1] == 66
assert X_v1_fit.shape[0] == 246008
assert X_v1_val.shape[0] == 61503

Full V1 matrix: (307511, 66)
V1 training split: (246008, 66)
V1 validation split: (61503, 66)
V1 test matrix: (48744, 66)

ANNUITY_TO_CREDIT included: True
AMT_CREDIT used as model feature: False
TARGET accidentally included: False
SK_ID_CURR accidentally included: False


## 8. V1 Numerical Preprocessing

Numerical transformations are fitted using the fixed training split only.
The same fitted thresholds and bin boundaries are then applied to the
validation and test sets to prevent data leakage.

In [12]:
# ---------------------------------------------------------
# V1 numerical preprocessing
# All learned thresholds are estimated from training only.
# ---------------------------------------------------------

# Work on independent copies
X_v1_fit_num = X_v1_fit.copy()
X_v1_val_num = X_v1_val.copy()
X_v1_test_num = X_v1_test.copy()


# 1. DAYS_EMPLOYED sentinel -> missing
for df in [X_v1_fit_num, X_v1_val_num, X_v1_test_num]:
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)


# 2. Learn P99.9 caps from TRAINING data only
cap_thresholds = {}

for col in cap_features:
    cap_thresholds[col] = X_v1_fit_num[col].quantile(0.999)

print("Training-fitted P99.9 thresholds:")
for col, threshold in cap_thresholds.items():
    print(f"{col}: {threshold}")


# Apply the same training-fitted caps to all datasets
for col, threshold in cap_thresholds.items():
    X_v1_fit_num[col] = X_v1_fit_num[col].clip(upper=threshold)
    X_v1_val_num[col] = X_v1_val_num[col].clip(upper=threshold)
    X_v1_test_num[col] = X_v1_test_num[col].clip(upper=threshold)


# 3. log1p transformation
# Part B specifies AMT_INCOME_TOTAL only
for df in [X_v1_fit_num, X_v1_val_num, X_v1_test_num]:
    df["AMT_INCOME_TOTAL"] = np.log1p(df["AMT_INCOME_TOTAL"])


# 4. Learn 10-quantile bin boundaries from TRAINING only
bin_edges = {}

for col in sparse_cont_features:
    non_missing = X_v1_fit_num[col].dropna()

    _, edges = pd.qcut(
        non_missing,
        q=10,
        retbins=True,
        duplicates="drop"
    )

    edges = np.unique(edges.astype(float))

    # Allow validation/test observations outside the training range
    edges[0] = -np.inf
    edges[-1] = np.inf

    bin_edges[col] = edges


# Apply missing indicators and the same bin edges
for col in sparse_cont_features:

    missing_col = f"{col}__MISSING"

    for df in [X_v1_fit_num, X_v1_val_num, X_v1_test_num]:

        # Missing-value indicator
        df[missing_col] = df[col].isna().astype("int8")

        # Replace the original sparse continuous value
        # with its training-fitted quantile-bin number
        df[col] = pd.cut(
            df[col],
            bins=bin_edges[col],
            labels=False,
            include_lowest=True
        )

        # Missing observations get bin code -1
        df[col] = df[col].fillna(-1).astype("int16")


print("\nNumerical preprocessing completed.")
print("Training shape:", X_v1_fit_num.shape)
print("Validation shape:", X_v1_val_num.shape)
print("Test shape:", X_v1_test_num.shape)

print("\nNumber of fitted continuous binning rules:", len(bin_edges))

Training-fitted P99.9 thresholds:
AMT_INCOME_TOTAL: 900000.0
AMT_REQ_CREDIT_BUREAU_QRT: 4.0
OBS_30_CNT_SOCIAL_CIRCLE: 17.0

Numerical preprocessing completed.
Training shape: (246008, 79)
Validation shape: (61503, 79)
Test shape: (48744, 79)

Number of fitted continuous binning rules: 13


## 9. V1 Categorical Preprocessing

Categorical transformations are fitted using the fixed training split only.
Rare categories are identified from training data only. High-cardinality
features use cross-fitted target encoding, while the remaining categorical
features use ordinal label encoding with safe handling of unseen categories.

In [13]:
from sklearn.preprocessing import TargetEncoder, OrdinalEncoder

# Start from the numerically processed V1 data
X_v1_fit_ready = X_v1_fit_num.copy()
X_v1_val_ready = X_v1_val_num.copy()
X_v1_test_ready = X_v1_test_num.copy()

# ---------------------------------------------------------
# 1. Missing indicators for Part B's sparse categoricals
# ---------------------------------------------------------

for col in sparse_cat_features:
    missing_col = f"{col}__MISSING"

    X_v1_fit_ready[missing_col] = (
        X_v1_fit_ready[col].isna().astype("int8")
    )
    X_v1_val_ready[missing_col] = (
        X_v1_val_ready[col].isna().astype("int8")
    )
    X_v1_test_ready[missing_col] = (
        X_v1_test_ready[col].isna().astype("int8")
    )


# ---------------------------------------------------------
# 2. Rare-category merging
#    Part B's final rule: frequency < 0.5% -> Other
#    Categories are learned from TRAINING only.
# ---------------------------------------------------------

categorical_features = (
    high_card_features
    + label_features
    + sparse_cat_features
)

rare_threshold = 0.005
rare_categories = {}

for col in categorical_features:

    train_col = (
        X_v1_fit_ready[col]
        .astype("string")
        .fillna("__MISSING__")
    )

    frequencies = train_col.value_counts(normalize=True)

    rare_categories[col] = set(
        frequencies[frequencies < rare_threshold].index
    )


def apply_category_rules(df, columns, rare_map):
    out = df.copy()

    for col in columns:

        s = (
            out[col]
            .astype("string")
            .fillna("__MISSING__")
        )

        s = s.where(
            ~s.isin(rare_map[col]),
            "__OTHER__"
        )

        out[col] = s.astype(str)

    return out


X_v1_fit_ready = apply_category_rules(
    X_v1_fit_ready,
    categorical_features,
    rare_categories
)

X_v1_val_ready = apply_category_rules(
    X_v1_val_ready,
    categorical_features,
    rare_categories
)

X_v1_test_ready = apply_category_rules(
    X_v1_test_ready,
    categorical_features,
    rare_categories
)


# ---------------------------------------------------------
# 3. Target encoding
#    OCCUPATION_TYPE and ORGANIZATION_TYPE
#
#    fit_transform uses cross-fitting on training data,
#    which reduces target leakage.
# ---------------------------------------------------------

target_encoder = TargetEncoder(
    target_type="binary",
    smooth="auto",
    cv=5,
    shuffle=True,
    random_state=42
)

X_v1_fit_ready[high_card_features] = (
    target_encoder.fit_transform(
        X_v1_fit_ready[high_card_features],
        y_v1_fit
    )
)

X_v1_val_ready[high_card_features] = (
    target_encoder.transform(
        X_v1_val_ready[high_card_features]
    )
)

X_v1_test_ready[high_card_features] = (
    target_encoder.transform(
        X_v1_test_ready[high_card_features]
    )
)


# ---------------------------------------------------------
# 4. Label / ordinal encoding
#    Includes ordinary label-encoded variables and
#    Part B's four sparse categorical variables.
# ---------------------------------------------------------

ordinal_features = label_features + sparse_cat_features

ordinal_encoder = OrdinalEncoder(
    handle_unknown="use_encoded_value",
    unknown_value=-1
)

X_v1_fit_ready[ordinal_features] = (
    ordinal_encoder.fit_transform(
        X_v1_fit_ready[ordinal_features]
    )
)

X_v1_val_ready[ordinal_features] = (
    ordinal_encoder.transform(
        X_v1_val_ready[ordinal_features]
    )
)

X_v1_test_ready[ordinal_features] = (
    ordinal_encoder.transform(
        X_v1_test_ready[ordinal_features]
    )
)


# ---------------------------------------------------------
# 5. Final checks
# ---------------------------------------------------------

remaining_object_cols = X_v1_fit_ready.select_dtypes(
    include=["object", "string", "category"]
).columns.tolist()

print("Final training shape:", X_v1_fit_ready.shape)
print("Final validation shape:", X_v1_val_ready.shape)
print("Final test shape:", X_v1_test_ready.shape)

print("\nTarget-encoded features:")
print(high_card_features)

print("\nOrdinal-encoded features:", len(ordinal_features))
print(ordinal_features)

print("\nRemaining non-numeric columns:")
print(remaining_object_cols)

print("\nTotal rare-category rules fitted:",
      len(rare_categories))

assert len(remaining_object_cols) == 0

Final training shape: (246008, 83)
Final validation shape: (61503, 83)
Final test shape: (48744, 83)

Target-encoded features:
['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']

Ordinal-encoded features: 14
['NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'CODE_GENDER', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'NAME_CONTRACT_TYPE', 'FLAG_OWN_CAR', 'NAME_TYPE_SUITE', 'WEEKDAY_APPR_PROCESS_START', 'FLAG_OWN_REALTY', 'EMERGENCYSTATE_MODE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE']

Remaining non-numeric columns:
[]

Total rare-category rules fitted: 16


## 10. V1 LightGBM Baseline

We evaluate the V1 feature set using exactly the same LightGBM configuration
as the V0 experiment. Keeping the model and validation split fixed allows us
to isolate the effect of Part B's feature selection and feature engineering.

In [13]:
# ---------------------------------------------------------
# V1 LightGBM baseline
# Use exactly the same model configuration as V0.
# ---------------------------------------------------------

v1_lgbm = LGBMClassifier(
    objective="binary",
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

# Train using the fixed training split
v1_lgbm.fit(
    X_v1_fit_ready,
    y_v1_fit
)

# Predict probabilities on the fixed validation split
v1_lgbm_val_prob = v1_lgbm.predict_proba(
    X_v1_val_ready
)[:, 1]

# ROC-AUC
v1_lgbm_auc = roc_auc_score(
    y_v1_val,
    v1_lgbm_val_prob
)

print(f"V0 Logistic Regression ROC-AUC: {v0_auc:.6f}")
print(f"V0 LightGBM ROC-AUC:            {v0_lgbm_auc:.6f}")
print(f"V1 LightGBM ROC-AUC:            {v1_lgbm_auc:.6f}")

print(
    f"\nV1 vs V0 LightGBM improvement: "
    f"{v1_lgbm_auc - v0_lgbm_auc:+.6f}"
)

V0 Logistic Regression ROC-AUC: 0.751239
V0 LightGBM ROC-AUC:            0.759364
V1 LightGBM ROC-AUC:            0.769522

V1 vs V0 LightGBM improvement: +0.010159


### Fixed-split results

Under the same fixed 80/20 validation protocol, LightGBM improved upon the
Logistic Regression baseline. Incorporating the V1 feature selection and
engineering rules from Part B produced a further improvement in ROC-AUC.

These fixed-split results are used for rapid model development; the next stage
uses stratified cross-validation to assess whether the improvement is stable.

In [14]:
results = pd.DataFrame([
    {
        "version": "V0",
        "model": "Logistic Regression",
        "feature_set": "V0",
        "validation": "Fixed 80/20",
        "roc_auc": v0_auc
    },
    {
        "version": "V0",
        "model": "LightGBM",
        "feature_set": "V0",
        "validation": "Fixed 80/20",
        "roc_auc": v0_lgbm_auc
    },
    {
        "version": "V1",
        "model": "LightGBM",
        "feature_set": "B V1",
        "validation": "Fixed 80/20",
        "roc_auc": v1_lgbm_auc
    }
])

results

,version,model,feature_set,validation,roc_auc
0,V0,Logistic Regression,V0,Fixed 80/20,0.751239
1,V0,LightGBM,V0,Fixed 80/20,0.759364
2,V1,LightGBM,B V1,Fixed 80/20,0.769522


## 11. Fold-Safe V1 Preprocessing for Cross-Validation

For cross-validation, all data-dependent preprocessing must be fitted
independently within each training fold. This prevents information from the
validation fold from influencing capping thresholds, quantile bins, rare
category definitions, or target encoding.

In [18]:
def prepare_v1_fold(X_train_raw, X_valid_raw, y_train_fold):
    """
    Fit V1 preprocessing using one training fold only,
    then apply the fitted rules to the corresponding validation fold.
    """

    # -----------------------------------------------------
    # 1. Build B's raw V1 features
    # -----------------------------------------------------
    X_tr = build_v1_raw_features(X_train_raw).copy()
    X_va = build_v1_raw_features(X_valid_raw).copy()

    # -----------------------------------------------------
    # 2. DAYS_EMPLOYED sentinel -> NaN
    # -----------------------------------------------------
    for df in [X_tr, X_va]:
        df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(
            365243, np.nan
        )

    # -----------------------------------------------------
    # 3. P99.9 capping learned from this training fold only
    # -----------------------------------------------------
    fold_caps = {}

    for col in cap_features:
        threshold = X_tr[col].quantile(0.999)
        fold_caps[col] = threshold

        X_tr[col] = X_tr[col].clip(upper=threshold)
        X_va[col] = X_va[col].clip(upper=threshold)

    # -----------------------------------------------------
    # 4. log1p
    # -----------------------------------------------------
    for col in log_features:
        X_tr[col] = np.log1p(X_tr[col])
        X_va[col] = np.log1p(X_va[col])

    # -----------------------------------------------------
    # 5. Sparse continuous:
    #    missing indicator + 10-quantile bins
    # -----------------------------------------------------
    fold_bin_edges = {}

    for col in sparse_cont_features:

        X_tr[f"{col}__MISSING"] = (
            X_tr[col].isna().astype("int8")
        )
        X_va[f"{col}__MISSING"] = (
            X_va[col].isna().astype("int8")
        )

        non_missing = X_tr[col].dropna()

        _, edges = pd.qcut(
            non_missing,
            q=10,
            retbins=True,
            duplicates="drop"
        )

        edges = np.unique(edges.astype(float))
        edges[0] = -np.inf
        edges[-1] = np.inf

        fold_bin_edges[col] = edges

        X_tr[col] = pd.cut(
            X_tr[col],
            bins=edges,
            labels=False,
            include_lowest=True
        ).fillna(-1).astype("int16")

        X_va[col] = pd.cut(
            X_va[col],
            bins=edges,
            labels=False,
            include_lowest=True
        ).fillna(-1).astype("int16")

    # -----------------------------------------------------
    # 6. Sparse categorical missing indicators
    # -----------------------------------------------------
    for col in sparse_cat_features:

        X_tr[f"{col}__MISSING"] = (
            X_tr[col].isna().astype("int8")
        )
        X_va[f"{col}__MISSING"] = (
            X_va[col].isna().astype("int8")
        )

    # -----------------------------------------------------
    # 7. Rare-category rules learned from training fold only
    # -----------------------------------------------------
    categorical_features = (
        high_card_features
        + label_features
        + sparse_cat_features
    )

    fold_rare = {}

    for col in categorical_features:

        s = (
            X_tr[col]
            .astype("string")
            .fillna("__MISSING__")
        )

        freq = s.value_counts(normalize=True)

        fold_rare[col] = set(
            freq[freq < 0.005].index
        )

    def apply_rare(df):

        out = df.copy()

        for col in categorical_features:

            s = (
                out[col]
                .astype("string")
                .fillna("__MISSING__")
            )

            s = s.where(
                ~s.isin(fold_rare[col]),
                "__OTHER__"
            )

            out[col] = s.astype(str)

        return out

    X_tr = apply_rare(X_tr)
    X_va = apply_rare(X_va)

    # -----------------------------------------------------
    # 8. Cross-fitted target encoding
    # -----------------------------------------------------
    fold_target_encoder = TargetEncoder(
        target_type="binary",
        smooth="auto",
        cv=5,
        shuffle=True,
        random_state=42
    )

    X_tr[high_card_features] = (
        fold_target_encoder.fit_transform(
            X_tr[high_card_features],
            y_train_fold
        )
    )

    X_va[high_card_features] = (
        fold_target_encoder.transform(
            X_va[high_card_features]
        )
    )

    # -----------------------------------------------------
    # 9. Ordinal encoding
    # -----------------------------------------------------
    ordinal_features = label_features + sparse_cat_features

    fold_ordinal_encoder = OrdinalEncoder(
        handle_unknown="use_encoded_value",
        unknown_value=-1
    )

    X_tr[ordinal_features] = (
        fold_ordinal_encoder.fit_transform(
            X_tr[ordinal_features]
        )
    )

    X_va[ordinal_features] = (
        fold_ordinal_encoder.transform(
            X_va[ordinal_features]
        )
    )

    # -----------------------------------------------------
    # Final safety checks
    # -----------------------------------------------------
    assert X_tr.shape[1] == X_va.shape[1]

    remaining = X_tr.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()

    assert len(remaining) == 0

    return X_tr, X_va

## 12. Five-Fold Stratified Cross-Validation

Five-fold stratified cross-validation is performed only within Part A's
predefined training partition. The original validation partition remains
untouched as a final holdout set.

For every fold, all V1 preprocessing steps are fitted using the fold-training
data only. LightGBM uses early stopping to determine the appropriate number
of boosting iterations.

In [16]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
import numpy as np
import lightgbm as lgb

# ---------------------------------------------------------
# Use ONLY Part A's predefined training partition for CV.
# The original 61,503-row validation set remains untouched.
# ---------------------------------------------------------

cv_raw = train.iloc[fit_idx].copy().reset_index(drop=True)
cv_y = cv_raw["TARGET"].astype(int).reset_index(drop=True)

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

fold_aucs = []
best_iterations = []

# Out-of-fold predictions for the whole 246,008-row training partition
oof_pred = np.zeros(len(cv_raw), dtype=float)

for fold, (fold_train_idx, fold_valid_idx) in enumerate(
    skf.split(cv_raw, cv_y),
    start=1
):

    print(f"\n========== Fold {fold} ==========")

    # Raw fold data
    X_fold_train_raw = cv_raw.iloc[fold_train_idx].copy()
    X_fold_valid_raw = cv_raw.iloc[fold_valid_idx].copy()

    y_fold_train = cv_y.iloc[fold_train_idx].copy()
    y_fold_valid = cv_y.iloc[fold_valid_idx].copy()

    # -----------------------------------------------------
    # Fit V1 preprocessing independently inside this fold
    # -----------------------------------------------------
    X_fold_train, X_fold_valid = prepare_v1_fold(
        X_fold_train_raw,
        X_fold_valid_raw,
        y_fold_train
    )

    print("Training shape:", X_fold_train.shape)
    print("Validation shape:", X_fold_valid.shape)

    # -----------------------------------------------------
    # LightGBM
    #
    # Same basic V1 model as before.
    # n_estimators=3000 is only an upper limit;
    # early stopping decides when to stop.
    # -----------------------------------------------------
    fold_model = LGBMClassifier(
        objective="binary",
        n_estimators=3000,
        learning_rate=0.05,
        num_leaves=31,
        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    fold_model.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            ),
            lgb.log_evaluation(period=0)
        ]
    )

    # Predict using the best iteration selected by early stopping
    fold_prob = fold_model.predict_proba(
        X_fold_valid,
        num_iteration=fold_model.best_iteration_
    )[:, 1]

    fold_auc = roc_auc_score(
        y_fold_valid,
        fold_prob
    )

    oof_pred[fold_valid_idx] = fold_prob
    fold_aucs.append(fold_auc)
    best_iterations.append(fold_model.best_iteration_)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Best iteration:      {fold_model.best_iteration_}")


# ---------------------------------------------------------
# CV summary
# ---------------------------------------------------------

mean_auc = np.mean(fold_aucs)
std_auc = np.std(fold_aucs, ddof=1)

oof_auc = roc_auc_score(
    cv_y,
    oof_pred
)

mean_best_iteration = int(
    round(np.mean(best_iterations))
)

print("\n===================================")
print("5-Fold CV Summary")
print("===================================")

for i, auc in enumerate(fold_aucs, start=1):
    print(f"Fold {i}: {auc:.6f}")

print(f"\nMean ROC-AUC:       {mean_auc:.6f}")
print(f"Std ROC-AUC:        {std_auc:.6f}")
print(f"OOF ROC-AUC:        {oof_auc:.6f}")

print("\nBest iterations:")
print(best_iterations)

print(
    "Mean best iteration:",
    mean_best_iteration
)


========== Fold 1 ==========
Training shape: (196806, 83)
Validation shape: (49202, 83)


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 1 ROC-AUC: 0.764139
Best iteration:      330

========== Fold 2 ==========
Training shape: (196806, 83)
Validation shape: (49202, 83)


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 2 ROC-AUC: 0.761383
Best iteration:      296

========== Fold 3 ==========
Training shape: (196806, 83)
Validation shape: (49202, 83)


...

Fold 3 ROC-AUC: 0.765526
Best iteration:      409

========== Fold 4 ==========
Training shape: (196807, 83)
Validation shape: (49201, 83)


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 4 ROC-AUC: 0.766793
Best iteration:      256

========== Fold 5 ==========
Training shape: (196807, 83)
Validation shape: (49201, 83)


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Fold 5 ROC-AUC: 0.760936
Best iteration:      179

5-Fold CV Summary
Fold 1: 0.764139
Fold 2: 0.761383
Fold 3: 0.765526
Fold 4: 0.766793
Fold 5: 0.760936

Mean ROC-AUC:       0.763755
Std ROC-AUC:        0.002554
OOF ROC-AUC:        0.763700

Best iterations:
[330, 296, 409, 256, 179]
Mean best iteration: 294


## 13. LightGBM Regularization Experiment

A regularized LightGBM configuration is evaluated using the same five-fold
stratified cross-validation protocol. The configuration is inspired by common
Home Credit LightGBM workflows, while all preprocessing and evaluation remain
specific to our V1 pipeline.

In [17]:
regularized_fold_aucs = []
regularized_best_iterations = []

for fold, (fold_train_idx, fold_valid_idx) in enumerate(
    skf.split(cv_raw, cv_y),
    start=1
):

    print(f"\n========== Regularized Fold {fold} ==========")

    X_fold_train_raw = cv_raw.iloc[fold_train_idx].copy()
    X_fold_valid_raw = cv_raw.iloc[fold_valid_idx].copy()

    y_fold_train = cv_y.iloc[fold_train_idx].copy()
    y_fold_valid = cv_y.iloc[fold_valid_idx].copy()

    # Fold-safe V1 preprocessing
    X_fold_train, X_fold_valid = prepare_v1_fold(
        X_fold_train_raw,
        X_fold_valid_raw,
        y_fold_train
    )

    # Regularized LightGBM candidate
    model_reg = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=32,

        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,

        reg_alpha=0.1,
        reg_lambda=0.1,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model_reg.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            ),
            lgb.log_evaluation(period=0)
        ]
    )

    fold_prob = model_reg.predict_proba(
        X_fold_valid,
        num_iteration=model_reg.best_iteration_
    )[:, 1]

    fold_auc = roc_auc_score(
        y_fold_valid,
        fold_prob
    )

    regularized_fold_aucs.append(fold_auc)
    regularized_best_iterations.append(
        model_reg.best_iteration_
    )

    print(f"ROC-AUC:        {fold_auc:.6f}")
    print(f"Best iteration: {model_reg.best_iteration_}")


regularized_mean_auc = np.mean(regularized_fold_aucs)
regularized_std_auc = np.std(
    regularized_fold_aucs,
    ddof=1
)

print("\n===================================")
print("Regularized LightGBM CV Summary")
print("===================================")

for i, auc in enumerate(
    regularized_fold_aucs,
    start=1
):
    print(f"Fold {i}: {auc:.6f}")

print(
    f"\nBaseline CV Mean:    {mean_auc:.6f}"
)

print(
    f"Regularized CV Mean: {regularized_mean_auc:.6f}"
)

print(
    f"Regularized CV Std:  {regularized_std_auc:.6f}"
)

print(
    "Improvement:",
    f"{regularized_mean_auc - mean_auc:+.6f}"
)

print(
    "Best iterations:",
    regularized_best_iterations
)


========== Regularized Fold 1 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.765283
Best iteration: 659

========== Regularized Fold 2 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.764092
Best iteration: 738

========== Regularized Fold 3 ==========


...

ROC-AUC:        0.767151
Best iteration: 660

========== Regularized Fold 4 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.769403
Best iteration: 875

========== Regularized Fold 5 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.764531
Best iteration: 593

Regularized LightGBM CV Summary
Fold 1: 0.765283
Fold 2: 0.764092
Fold 3: 0.767151
Fold 4: 0.769403
Fold 5: 0.764531

Baseline CV Mean:    0.763755
Regularized CV Mean: 0.766092
Regularized CV Std:  0.002189
Improvement: +0.002336
Best iterations: [659, 738, 660, 875, 593]


## 14. Class-Imbalance Weighting Experiment

The best regularized LightGBM configuration is further evaluated with
class-imbalance weighting. The positive-class weight is calculated
independently within each training fold to avoid information leakage.

In [18]:
weighted_fold_aucs = []
weighted_best_iterations = []

for fold, (fold_train_idx, fold_valid_idx) in enumerate(
    skf.split(cv_raw, cv_y),
    start=1
):

    print(f"\n========== Weighted Fold {fold} ==========")

    X_fold_train_raw = cv_raw.iloc[fold_train_idx].copy()
    X_fold_valid_raw = cv_raw.iloc[fold_valid_idx].copy()

    y_fold_train = cv_y.iloc[fold_train_idx].copy()
    y_fold_valid = cv_y.iloc[fold_valid_idx].copy()

    # Fold-safe V1 preprocessing
    X_fold_train, X_fold_valid = prepare_v1_fold(
        X_fold_train_raw,
        X_fold_valid_raw,
        y_fold_train
    )

    # Calculate class weight using THIS training fold only
    n_negative = (y_fold_train == 0).sum()
    n_positive = (y_fold_train == 1).sum()

    fold_scale_pos_weight = (
        n_negative / n_positive
    )

    print(
        "scale_pos_weight:",
        round(fold_scale_pos_weight, 4)
    )

    model_weighted = LGBMClassifier(
        objective="binary",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=32,

        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,

        reg_alpha=0.1,
        reg_lambda=0.1,

        scale_pos_weight=fold_scale_pos_weight,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    model_weighted.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                verbose=False
            ),
            lgb.log_evaluation(period=0)
        ]
    )

    fold_prob = model_weighted.predict_proba(
        X_fold_valid,
        num_iteration=model_weighted.best_iteration_
    )[:, 1]

    fold_auc = roc_auc_score(
        y_fold_valid,
        fold_prob
    )

    weighted_fold_aucs.append(fold_auc)

    weighted_best_iterations.append(
        model_weighted.best_iteration_
    )

    print(f"ROC-AUC:        {fold_auc:.6f}")
    print(
        f"Best iteration: "
        f"{model_weighted.best_iteration_}"
    )


weighted_mean_auc = np.mean(
    weighted_fold_aucs
)

weighted_std_auc = np.std(
    weighted_fold_aucs,
    ddof=1
)

print("\n===================================")
print("Weighted LightGBM CV Summary")
print("===================================")

for i, auc in enumerate(
    weighted_fold_aucs,
    start=1
):
    print(f"Fold {i}: {auc:.6f}")

print(
    f"\nBaseline CV Mean:    "
    f"{mean_auc:.6f}"
)

print(
    f"Regularized CV Mean: "
    f"{regularized_mean_auc:.6f}"
)

print(
    f"Weighted CV Mean:    "
    f"{weighted_mean_auc:.6f}"
)

print(
    f"Weighted CV Std:     "
    f"{weighted_std_auc:.6f}"
)

print(
    "Weighted vs Regularized:",
    f"{weighted_mean_auc - regularized_mean_auc:+.6f}"
)

print(
    "Best iterations:",
    weighted_best_iterations
)


========== Weighted Fold 1 ==========
scale_pos_weight: 11.3871


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.715875
Best iteration: 2

========== Weighted Fold 2 ==========
scale_pos_weight: 11.3871


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.718883
Best iteration: 2

========== Weighted Fold 3 ==========
scale_pos_weight: 11.3871


...

ROC-AUC:        0.726534
Best iteration: 2

========== Weighted Fold 4 ==========
scale_pos_weight: 11.3871


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.726485
Best iteration: 2

========== Weighted Fold 5 ==========
scale_pos_weight: 11.3871


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


ROC-AUC:        0.715293
Best iteration: 2

Weighted LightGBM CV Summary
Fold 1: 0.715875
Fold 2: 0.718883
Fold 3: 0.726534
Fold 4: 0.726485
Fold 5: 0.715293

Baseline CV Mean:    0.763755
Regularized CV Mean: 0.766092
Weighted CV Mean:    0.720614
Weighted CV Std:     0.005552
Weighted vs Regularized: -0.045478
Best iterations: [2, 2, 2, 2, 2]


### Corrected Class-Weight Experiment

The previous weighted experiment stopped after only two boosting iterations
in every fold, suggesting that early stopping was influenced by an auxiliary
metric rather than ROC-AUC alone. We therefore repeat the experiment with
AUC explicitly set as the optimization metric for early stopping.

In [19]:
weighted_auc_fold_scores = []
weighted_auc_best_iterations = []

for fold, (fold_train_idx, fold_valid_idx) in enumerate(
    skf.split(cv_raw, cv_y),
    start=1
):

    print(f"\n========== Corrected Weighted Fold {fold} ==========")

    X_fold_train_raw = cv_raw.iloc[fold_train_idx].copy()
    X_fold_valid_raw = cv_raw.iloc[fold_valid_idx].copy()

    y_fold_train = cv_y.iloc[fold_train_idx].copy()
    y_fold_valid = cv_y.iloc[fold_valid_idx].copy()

    # Fold-safe V1 preprocessing
    X_fold_train, X_fold_valid = prepare_v1_fold(
        X_fold_train_raw,
        X_fold_valid_raw,
        y_fold_train
    )

    # Class imbalance ratio learned from this training fold only
    n_negative = (y_fold_train == 0).sum()
    n_positive = (y_fold_train == 1).sum()

    fold_scale_pos_weight = n_negative / n_positive

    corrected_weighted_model = LGBMClassifier(
        objective="binary",
        metric="auc",

        n_estimators=3000,
        learning_rate=0.03,

        num_leaves=32,

        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,

        reg_alpha=0.1,
        reg_lambda=0.1,

        scale_pos_weight=fold_scale_pos_weight,

        random_state=42,
        n_jobs=-1,
        verbosity=-1
    )

    corrected_weighted_model.fit(
        X_fold_train,
        y_fold_train,
        eval_set=[(X_fold_valid, y_fold_valid)],
        eval_metric="auc",
        callbacks=[
            lgb.early_stopping(
                stopping_rounds=100,
                first_metric_only=True,
                verbose=False
            ),
            lgb.log_evaluation(period=0)
        ]
    )

    fold_prob = corrected_weighted_model.predict_proba(
        X_fold_valid,
        num_iteration=corrected_weighted_model.best_iteration_
    )[:, 1]

    fold_auc = roc_auc_score(
        y_fold_valid,
        fold_prob
    )

    weighted_auc_fold_scores.append(fold_auc)
    weighted_auc_best_iterations.append(
        corrected_weighted_model.best_iteration_
    )

    print(f"scale_pos_weight: {fold_scale_pos_weight:.4f}")
    print(f"ROC-AUC:          {fold_auc:.6f}")
    print(
        f"Best iteration:   "
        f"{corrected_weighted_model.best_iteration_}"
    )


weighted_auc_mean = np.mean(
    weighted_auc_fold_scores
)

weighted_auc_std = np.std(
    weighted_auc_fold_scores,
    ddof=1
)

print("\n===================================")
print("Corrected Weighted CV Summary")
print("===================================")

for i, auc in enumerate(
    weighted_auc_fold_scores,
    start=1
):
    print(f"Fold {i}: {auc:.6f}")

print(
    f"\nRegularized CV Mean: "
    f"{regularized_mean_auc:.6f}"
)

print(
    f"Weighted CV Mean:    "
    f"{weighted_auc_mean:.6f}"
)

print(
    f"Weighted CV Std:     "
    f"{weighted_auc_std:.6f}"
)

print(
    "Weighted vs Regularized:",
    f"{weighted_auc_mean - regularized_mean_auc:+.6f}"
)

print(
    "Best iterations:",
    weighted_auc_best_iterations
)


========== Corrected Weighted Fold 1 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


scale_pos_weight: 11.3871
ROC-AUC:          0.764746
Best iteration:   482

========== Corrected Weighted Fold 2 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


scale_pos_weight: 11.3871
ROC-AUC:          0.763837
Best iteration:   586

========== Corrected Weighted Fold 3 ==========


...

scale_pos_weight: 11.3871
ROC-AUC:          0.766848
Best iteration:   618

========== Corrected Weighted Fold 4 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


scale_pos_weight: 11.3871
ROC-AUC:          0.768731
Best iteration:   453

========== Corrected Weighted Fold 5 ==========


e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


scale_pos_weight: 11.3871
ROC-AUC:          0.763588
Best iteration:   570

Corrected Weighted CV Summary
Fold 1: 0.764746
Fold 2: 0.763837
Fold 3: 0.766848
Fold 4: 0.768731
Fold 5: 0.763588

Regularized CV Mean: 0.766092
Weighted CV Mean:    0.765550
Weighted CV Std:     0.002193
Weighted vs Regularized: -0.000542
Best iterations: [482, 586, 618, 453, 570]


## 15. Final Model Selection on the Fixed Validation Split

Based on five-fold cross-validation, the regularized V1 LightGBM model was
selected as the final configuration. Class weighting was not retained because
it did not improve mean cross-validation ROC-AUC.

The selected configuration is now evaluated on Part A's predefined validation
split for consistent comparison with the earlier V0 and V1 experiments.

In [20]:
# ---------------------------------------------------------
# Final selected LightGBM configuration
# ---------------------------------------------------------

final_selected_model = LGBMClassifier(
    objective="binary",
    metric="auc",

    n_estimators=3000,
    learning_rate=0.03,

    num_leaves=32,

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

final_selected_model.fit(
    X_v1_fit_ready,
    y_v1_fit,
    eval_set=[(X_v1_val_ready, y_v1_val)],
    eval_metric="auc",
    callbacks=[
        lgb.early_stopping(
            stopping_rounds=100,
            first_metric_only=True,
            verbose=False
        ),
        lgb.log_evaluation(period=0)
    ]
)

final_val_prob = final_selected_model.predict_proba(
    X_v1_val_ready,
    num_iteration=final_selected_model.best_iteration_
)[:, 1]

final_val_auc = roc_auc_score(
    y_v1_val,
    final_val_prob
)

print("===================================")
print("Final Fixed-Split Evaluation")
print("===================================")

print(f"V0 Logistic Regression:   {v0_auc:.6f}")
print(f"V0 LightGBM:              {v0_lgbm_auc:.6f}")
print(f"V1 LightGBM baseline:     {v1_lgbm_auc:.6f}")
print(f"Final Regularized V1:     {final_val_auc:.6f}")

print(
    "\nImprovement over V1 baseline:",
    f"{final_val_auc - v1_lgbm_auc:+.6f}"
)

print(
    "Improvement over V0 LightGBM:",
    f"{final_val_auc - v0_lgbm_auc:+.6f}"
)

print(
    "Best iteration:",
    final_selected_model.best_iteration_
)

e:\MSDM5054\msdm5054-home-credit\.venv\Lib\site-packages\lightgbm\sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


Final Fixed-Split Evaluation
V0 Logistic Regression:   0.751239
V0 LightGBM:              0.759364
V1 LightGBM baseline:     0.769522
Final Regularized V1:     0.772396

Improvement over V1 baseline: +0.002874
Improvement over V0 LightGBM: +0.013032
Best iteration: 892


## 16. Model Selection Summary

Model development progressed from the original Logistic Regression baseline
to LightGBM, followed by the V1 feature-engineering pipeline and regularized
LightGBM tuning.

Five-fold stratified cross-validation was used to assess model stability.
Class weighting was also tested but was not retained because it did not
improve mean ROC-AUC.

The regularized V1 LightGBM configuration was selected as the final model.

In [21]:
model_summary = pd.DataFrame([
    {
        "experiment": "V0 Logistic Regression",
        "validation": "Fixed 80/20",
        "roc_auc": v0_auc
    },
    {
        "experiment": "V0 LightGBM",
        "validation": "Fixed 80/20",
        "roc_auc": v0_lgbm_auc
    },
    {
        "experiment": "V1 LightGBM baseline",
        "validation": "Fixed 80/20",
        "roc_auc": v1_lgbm_auc
    },
    {
        "experiment": "V1 LightGBM baseline",
        "validation": "5-fold CV mean",
        "roc_auc": mean_auc
    },
    {
        "experiment": "V1 Regularized LightGBM",
        "validation": "5-fold CV mean",
        "roc_auc": regularized_mean_auc
    },
    {
        "experiment": "V1 Regularized + class weight",
        "validation": "5-fold CV mean",
        "roc_auc": weighted_auc_mean
    },
    {
        "experiment": "Final V1 Regularized LightGBM",
        "validation": "Fixed 80/20",
        "roc_auc": final_val_auc
    }
])

model_summary["roc_auc"] = model_summary["roc_auc"].round(6)

model_summary

,experiment,validation,roc_auc
0,V0 Logistic Regression,Fixed 80/20,0.751239
1,V0 LightGBM,Fixed 80/20,0.759364
2,V1 LightGBM baseline,Fixed 80/20,0.769522
3,V1 LightGBM baseline,5-fold CV mean,0.763755
4,V1 Regularized LightGBM,5-fold CV mean,0.766092
5,V1 Regularized + class weight,5-fold CV mean,0.765550
6,Final V1 Regularized LightGBM,Fixed 80/20,0.772396


## 17. Final Full-Data Training and Test Prediction

After model selection, the final V1 preprocessing pipeline is fitted using
all available labelled training observations. The selected regularized
LightGBM configuration is then retrained on the full training set.

The number of boosting iterations is set to 705, the mean best iteration
obtained from the five-fold cross-validation experiment. Predictions are
generated for the official Kaggle test set.

In [22]:
# ---------------------------------------------------------
# Final full-data V1 preprocessing
# ---------------------------------------------------------

X_full_raw = train.copy()
y_full = train["TARGET"].astype(int).copy()

X_test_raw = test.copy()

X_full_ready, X_test_ready = prepare_v1_fold(
    X_full_raw,
    X_test_raw,
    y_full
)

print("Final full training matrix:", X_full_ready.shape)
print("Final test matrix:", X_test_ready.shape)

print(
    "Remaining non-numeric training columns:",
    X_full_ready.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
)

print(
    "Remaining non-numeric test columns:",
    X_test_ready.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
)

assert X_full_ready.shape[0] == 307511
assert X_test_ready.shape[0] == 48744
assert X_full_ready.shape[1] == X_test_ready.shape[1]

Final full training matrix: (307511, 83)
Final test matrix: (48744, 83)
Remaining non-numeric training columns: []
Remaining non-numeric test columns: []


In [23]:
# ---------------------------------------------------------
# Train final LightGBM on all labelled training data
# ---------------------------------------------------------

FINAL_N_ESTIMATORS = int(
    round(np.mean(regularized_best_iterations))
)

print(
    "Final number of estimators from CV:",
    FINAL_N_ESTIMATORS
)

final_full_model = LGBMClassifier(
    objective="binary",

    n_estimators=FINAL_N_ESTIMATORS,
    learning_rate=0.03,

    num_leaves=32,

    subsample=0.8,
    subsample_freq=1,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=0.1,

    random_state=42,
    n_jobs=-1,
    verbosity=-1
)

final_full_model.fit(
    X_full_ready,
    y_full
)

print("Final full-data model fitted successfully.")

Final number of estimators from CV: 705
Final full-data model fitted successfully.


In [24]:
# ---------------------------------------------------------
# Predict official Kaggle test data
# ---------------------------------------------------------

final_test_prob = final_full_model.predict_proba(
    X_test_ready
)[:, 1]

print("Number of test predictions:", len(final_test_prob))
print("Minimum probability:", final_test_prob.min())
print("Maximum probability:", final_test_prob.max())
print("Mean probability:", final_test_prob.mean())

assert len(final_test_prob) == 48744
assert np.isfinite(final_test_prob).all()
assert ((final_test_prob >= 0) & (final_test_prob <= 1)).all()

Number of test predictions: 48744
Minimum probability: 0.002934770748304166
Maximum probability: 0.7409438159571201
Mean probability: 0.07910378917535556


## 18. Final Submission File

The final full-data LightGBM model is used to generate default-risk
probabilities for the official test set. The submission file follows the
required Home Credit format with exactly two columns: SK_ID_CURR and TARGET.

The file is validated locally before any manual Kaggle submission.

In [25]:
# ---------------------------------------------------------
# Create and validate final V1 submission
# ---------------------------------------------------------

from pathlib import Path
import sys

sys.path.insert(0, str(ROOT / "src"))
from validate_submission import validate

submission_dir = ROOT / "results" / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

submission_v1_path = submission_dir / "submission_v1.csv"

# Avoid accidentally overwriting an existing result
if submission_v1_path.exists():
    raise FileExistsError(
        f"{submission_v1_path} already exists. "
        "Compare the existing file before replacing it."
    )

submission_v1 = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"].to_numpy(),
    "TARGET": final_test_prob
})

# Load official Kaggle template
sample_submission = pd.read_csv(
    ROOT / "data" / "raw" / "sample_submission.csv"
)

# ---------------------------------------------------------
# Basic integrity checks
# ---------------------------------------------------------

assert submission_v1.shape == (48744, 2)

assert submission_v1.columns.tolist() == [
    "SK_ID_CURR",
    "TARGET"
]

assert submission_v1["SK_ID_CURR"].is_unique

assert (
    submission_v1["SK_ID_CURR"].to_numpy()
    ==
    sample_submission["SK_ID_CURR"].to_numpy()
).all()

assert np.isfinite(
    submission_v1["TARGET"]
).all()

assert submission_v1["TARGET"].between(
    0, 1
).all()

# Save
submission_v1.to_csv(
    submission_v1_path,
    index=False
)

print("Saved to:")
print(submission_v1_path)

print("\nSubmission shape:")
print(submission_v1.shape)

print("\nFirst five rows:")
display(submission_v1.head())

print("\nValidation:")
print(
    validate(
        submission_v1_path,
        ROOT / "data/raw/sample_submission.csv",
        ROOT / "data/raw/application_test.csv"
    )
)

Saved to:
e:\MSDM5054\msdm5054-home-credit\results\submissions\submission_v1.csv

Submission shape:
(48744, 2)

First five rows:


,SK_ID_CURR,TARGET
0,100001,0.027853
1,100005,0.101450
2,100013,0.013031
3,100028,0.039106
4,100038,0.162166



Validation:
PASS: 48744 rows, unique matching IDs, correct columns, valid probabilities; nothing submitted.


## 19. C-Part Outputs and Handoff

The selected model, validation predictions, cross-validation results,
feature importance, model configuration and experiment summary are exported
for downstream evaluation and report integration.

The saved full-data LightGBM model expects the already-preprocessed V1
83-feature matrix rather than the original raw application table.

In [26]:
# ---------------------------------------------------------
# Export C-part results for downstream handoff
# ---------------------------------------------------------

import json
import joblib

C_RESULTS = ROOT / "results" / "c"
C_RESULTS.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. Model comparison summary
# =========================================================

model_summary_path = C_RESULTS / "model_summary.csv"

model_summary.to_csv(
    model_summary_path,
    index=False
)


# =========================================================
# 2. Final fixed-validation predictions
#    Useful for Part D: ROC, PR curve, calibration, etc.
# =========================================================

validation_predictions = pd.DataFrame({
    "SK_ID_CURR": train.iloc[val_idx]["SK_ID_CURR"].to_numpy(),
    "TARGET": y_v1_val.to_numpy(),
    "PRED_PROB": final_val_prob
})

validation_predictions_path = (
    C_RESULTS / "validation_predictions_v1.csv"
)

validation_predictions.to_csv(
    validation_predictions_path,
    index=False
)

assert validation_predictions.shape[0] == 61503
assert validation_predictions["SK_ID_CURR"].is_unique
assert np.isfinite(
    validation_predictions["PRED_PROB"]
).all()


# =========================================================
# 3. Five-fold CV results
# =========================================================

cv_results = pd.DataFrame({
    "fold": np.arange(1, 6),
    "baseline_v1_auc": fold_aucs,
    "regularized_v1_auc": regularized_fold_aucs,
    "weighted_v1_auc": weighted_auc_fold_scores,
    "regularized_best_iteration": regularized_best_iterations
})

cv_results_path = C_RESULTS / "cv_results.csv"

cv_results.to_csv(
    cv_results_path,
    index=False
)


# =========================================================
# 4. Final model configuration
# =========================================================

final_config = {
    "model": "LightGBM",
    "feature_version": "V1",
    "feature_count": int(X_full_ready.shape[1]),

    "parameters": {
        "objective": "binary",
        "n_estimators": int(FINAL_N_ESTIMATORS),
        "learning_rate": 0.03,
        "num_leaves": 32,
        "subsample": 0.8,
        "subsample_freq": 1,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 0.1,
        "random_state": 42
    },

    "fixed_validation_auc": float(final_val_auc),

    "regularized_cv_mean_auc": float(
        regularized_mean_auc
    ),

    "regularized_cv_std_auc": float(
        regularized_std_auc
    ),

    "cv_best_iterations": [
        int(x) for x in regularized_best_iterations
    ],

    "final_n_estimators": int(
        FINAL_N_ESTIMATORS
    ),

    "class_weight_used": False
}

config_path = C_RESULTS / "final_model_config.json"

with open(
    config_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        final_config,
        f,
        indent=2
    )


# =========================================================
# 5. Feature importance
# =========================================================

feature_importance = pd.DataFrame({
    "feature": X_full_ready.columns,
    "importance": final_full_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

feature_importance_path = (
    C_RESULTS / "feature_importance.csv"
)

feature_importance.to_csv(
    feature_importance_path,
    index=False
)


# =========================================================
# 6. Exact model feature order
# =========================================================

feature_names_path = (
    C_RESULTS / "final_feature_names.json"
)

with open(
    feature_names_path,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        X_full_ready.columns.tolist(),
        f,
        indent=2
    )


# =========================================================
# 7. Save fitted final LightGBM model
# =========================================================

model_path = (
    C_RESULTS /
    "final_lightgbm_preprocessed_input.joblib"
)

joblib.dump(
    final_full_model,
    model_path
)


# =========================================================
# Final check
# =========================================================

print("C-part handoff files created:\n")

for path in [
    model_summary_path,
    validation_predictions_path,
    cv_results_path,
    config_path,
    feature_importance_path,
    feature_names_path,
    model_path
]:
    print("-", path.relative_to(ROOT))

C-part handoff files created:

- results\c\model_summary.csv
- results\c\validation_predictions_v1.csv
- results\c\cv_results.csv
- results\c\final_model_config.json
- results\c\feature_importance.csv
- results\c\final_feature_names.json
- results\c\final_lightgbm_preprocessed_input.joblib


## 20. Optional Enhancement: XGBoost Comparison

To strengthen the model comparison, XGBoost is evaluated as an additional
gradient-boosting model under the same V1 feature representation and validation
protocol used for LightGBM.

The purpose of this experiment is not to replace the selected LightGBM model
automatically, but to determine whether the final model choice remains
competitive when compared with another strong boosting algorithm.

### 20.1 Fixed-Split XGBoost Sanity Check

Before running the full five-fold cross-validation, XGBoost is first
evaluated on the predefined 80/20 split using exactly the same V1
preprocessed feature matrices as the LightGBM experiments.

This fixed-split result is used as a sanity check and a directly comparable
reference, while the final model comparison will rely primarily on
cross-validation.

In [16]:
# ---------------------------------------------------------
# XGBoost fixed-split sanity check
# ---------------------------------------------------------

import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import roc_auc_score

print("XGBoost version:", xgb.__version__)

xgb_fixed_model = XGBClassifier(
    objective="binary:logistic",

    n_estimators=3000,
    learning_rate=0.03,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    tree_method="hist",

    eval_metric="auc",
    early_stopping_rounds=100,

    random_state=42,
    n_jobs=-1,
    verbosity=0
)

xgb_fixed_model.fit(
    X_v1_fit_ready,
    y_v1_fit,
    eval_set=[
        (X_v1_val_ready, y_v1_val)
    ],
    verbose=False
)

xgb_fixed_val_prob = xgb_fixed_model.predict_proba(
    X_v1_val_ready
)[:, 1]

xgb_fixed_auc = roc_auc_score(
    y_v1_val,
    xgb_fixed_val_prob
)

print(f"XGBoost fixed validation AUC: {xgb_fixed_auc:.6f}")
print(
    "XGBoost best iteration:",
    xgb_fixed_model.best_iteration
)

print()
print("Reference:")
print("V1 LightGBM baseline:       0.769522")
print("Regularized LightGBM:       0.772396")
print(f"XGBoost:                    {xgb_fixed_auc:.6f}")

XGBoost version: 3.4.1
XGBoost fixed validation AUC: 0.772394
XGBoost best iteration: 777

Reference:
V1 LightGBM baseline:       0.769522
Regularized LightGBM:       0.772396
XGBoost:                    0.772394


### 20.2 Leakage-Safe Five-Fold Cross-Validation for XGBoost

To provide a fair comparison with the selected LightGBM model, XGBoost is
evaluated using the same five-fold stratified cross-validation framework.

For every fold, all data-dependent preprocessing steps are fitted only on
the corresponding training fold and then applied to the validation fold.
This includes capping, quantile binning, rare-category handling, target
encoding and ordinal encoding.

Therefore, the XGBoost and LightGBM cross-validation results are directly
comparable under the same V1 feature and validation protocol.

In [20]:
# ---------------------------------------------------------
# Leakage-safe 5-fold CV for XGBoost
# ---------------------------------------------------------

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier

# Use exactly the same 80% training partition used previously
X_xgb_cv_raw = train.iloc[fit_idx].copy()
y_xgb_cv = train.iloc[fit_idx]["TARGET"].astype(int).copy()

xgb_cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

xgb_fold_aucs = []
xgb_best_iterations = []

xgb_oof_prob = np.zeros(
    len(X_xgb_cv_raw),
    dtype=float
)

for fold, (tr_idx, va_idx) in enumerate(
    xgb_cv.split(X_xgb_cv_raw, y_xgb_cv),
    start=1
):

    print(f"\n========== XGBoost Fold {fold} / 5 ==========")

    X_tr_raw = X_xgb_cv_raw.iloc[tr_idx].copy()
    X_va_raw = X_xgb_cv_raw.iloc[va_idx].copy()

    y_tr = y_xgb_cv.iloc[tr_idx].copy()
    y_va = y_xgb_cv.iloc[va_idx].copy()

    # -----------------------------------------
    # Fold-safe V1 preprocessing
    # -----------------------------------------

    X_tr_ready, X_va_ready = prepare_v1_fold(
        X_tr_raw,
        X_va_raw,
        y_tr
    )

    # -----------------------------------------
    # XGBoost
    # -----------------------------------------

    model = XGBClassifier(
        objective="binary:logistic",

        n_estimators=3000,
        learning_rate=0.03,
        max_depth=6,

        subsample=0.8,
        colsample_bytree=0.8,

        reg_alpha=0.1,
        reg_lambda=1.0,

        tree_method="hist",

        eval_metric="auc",
        early_stopping_rounds=100,

        random_state=42,
        n_jobs=-1,
        verbosity=0
    )

    model.fit(
        X_tr_ready,
        y_tr,
        eval_set=[
            (X_va_ready, y_va)
        ],
        verbose=False
    )

    va_prob = model.predict_proba(
        X_va_ready
    )[:, 1]

    fold_auc = roc_auc_score(
        y_va,
        va_prob
    )

    # XGBoost best_iteration is zero-based
    best_rounds = model.best_iteration + 1

    xgb_fold_aucs.append(fold_auc)
    xgb_best_iterations.append(best_rounds)

    xgb_oof_prob[va_idx] = va_prob

    print(
        f"Fold {fold} AUC: "
        f"{fold_auc:.6f}"
    )

    print(
        f"Best boosting rounds: "
        f"{best_rounds}"
    )


# ---------------------------------------------------------
# CV summary
# ---------------------------------------------------------

xgb_cv_mean_auc = np.mean(
    xgb_fold_aucs
)

xgb_cv_std_auc = np.std(
    xgb_fold_aucs
)

xgb_oof_auc = roc_auc_score(
    y_xgb_cv,
    xgb_oof_prob
)

print("\n==========================================")
print("XGBoost 5-fold CV summary")
print("==========================================")

print(
    "Fold AUCs:",
    [
        round(x, 6)
        for x in xgb_fold_aucs
    ]
)

print(
    f"Mean AUC: {xgb_cv_mean_auc:.6f}"
)

print(
    f"Std AUC:  {xgb_cv_std_auc:.6f}"
)

print(
    f"OOF AUC:  {xgb_oof_auc:.6f}"
)

print(
    "Best boosting rounds:",
    xgb_best_iterations
)

print(
    "Mean best boosting rounds:",
    int(round(
        np.mean(xgb_best_iterations)
    ))
)

print("\nLightGBM reference:")
print(
    "Regularized LightGBM CV mean: "
    "0.766092"
)
print(
    "Regularized LightGBM CV std:  "
    "0.002189"
)

print(
    f"XGBoost - LightGBM mean AUC: "
    f"{xgb_cv_mean_auc - 0.766092:+.6f}"
)


========== XGBoost Fold 1 / 5 ==========
Fold 1 AUC: 0.767116
Best boosting rounds: 694

========== XGBoost Fold 2 / 5 ==========
Fold 2 AUC: 0.764908
Best boosting rounds: 477

========== XGBoost Fold 3 / 5 ==========
Fold 3 AUC: 0.768490
Best boosting rounds: 582

========== XGBoost Fold 4 / 5 ==========
Fold 4 AUC: 0.768884
Best boosting rounds: 793

========== XGBoost Fold 5 / 5 ==========
Fold 5 AUC: 0.766653
Best boosting rounds: 805

XGBoost 5-fold CV summary
Fold AUCs: [0.767116, 0.764908, 0.76849, 0.768884, 0.766653]
Mean AUC: 0.767210
Std AUC:  0.001418
OOF AUC:  0.767191
Best boosting rounds: [694, 477, 582, 793, 805]
Mean best boosting rounds: 670

LightGBM reference:
Regularized LightGBM CV mean: 0.766092
Regularized LightGBM CV std:  0.002189
XGBoost - LightGBM mean AUC: +0.001118


In [21]:
print("XGBoost fold AUCs:", [round(x, 6) for x in xgb_fold_aucs])
print(f"XGBoost CV mean: {xgb_cv_mean_auc:.6f}")
print(f"XGBoost CV std:  {xgb_cv_std_auc:.6f}")
print(f"XGBoost OOF AUC: {xgb_oof_auc:.6f}")
print("Best boosting rounds:", xgb_best_iterations)
print(
    "Mean best boosting rounds:",
    int(round(np.mean(xgb_best_iterations)))
)

XGBoost fold AUCs: [0.767116, 0.764908, 0.76849, 0.768884, 0.766653]
XGBoost CV mean: 0.767210
XGBoost CV std:  0.001418
XGBoost OOF AUC: 0.767191
Best boosting rounds: [694, 477, 582, 793, 805]
Mean best boosting rounds: 670


### 20.3 Final XGBoost Training on Full Data

Based on five-fold cross-validation, XGBoost achieved a slightly higher
mean ROC-AUC and lower fold-to-fold variability than the regularized
LightGBM model.

The final XGBoost model is therefore retrained using all labelled training
observations. The number of boosting rounds is set to 670, corresponding
to the mean best iteration obtained from five-fold cross-validation.

In [22]:
# ---------------------------------------------------------
# Full-data preprocessing for final XGBoost
# ---------------------------------------------------------

X_full_raw = train.copy()
y_full = train["TARGET"].astype(int).copy()

X_test_raw = test.copy()

X_full_ready, X_test_ready = prepare_v1_fold(
    X_full_raw,
    X_test_raw,
    y_full
)

print("Full training matrix:", X_full_ready.shape)
print("Test matrix:", X_test_ready.shape)

print(
    "Remaining non-numeric training columns:",
    X_full_ready.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
)

print(
    "Remaining non-numeric test columns:",
    X_test_ready.select_dtypes(
        include=["object", "string", "category"]
    ).columns.tolist()
)

assert X_full_ready.shape == (307511, 83)
assert X_test_ready.shape == (48744, 83)

Full training matrix: (307511, 83)
Test matrix: (48744, 83)
Remaining non-numeric training columns: []
Remaining non-numeric test columns: []


### 20.4 Final XGBoost Model

The final XGBoost model is trained on the complete labelled training set
using the selected V1 feature representation.

The number of boosting rounds is fixed at 670, corresponding to the mean
best iteration obtained from five-fold cross-validation. No early stopping
is used during full-data training because no independent validation fold is
reserved at this stage.

In [23]:
# ---------------------------------------------------------
# Train final XGBoost model on all labelled data
# ---------------------------------------------------------

FINAL_XGB_N_ESTIMATORS = int(
    round(np.mean(xgb_best_iterations))
)

print(
    "Final XGBoost boosting rounds:",
    FINAL_XGB_N_ESTIMATORS
)

final_xgb_model = XGBClassifier(
    objective="binary:logistic",

    n_estimators=FINAL_XGB_N_ESTIMATORS,
    learning_rate=0.03,
    max_depth=6,

    subsample=0.8,
    colsample_bytree=0.8,

    reg_alpha=0.1,
    reg_lambda=1.0,

    tree_method="hist",
    eval_metric="auc",

    random_state=42,
    n_jobs=-1,
    verbosity=0
)

final_xgb_model.fit(
    X_full_ready,
    y_full
)

print(
    "Final full-data XGBoost model "
    "fitted successfully."
)

Final XGBoost boosting rounds: 670
Final full-data XGBoost model fitted successfully.


In [25]:
# ---------------------------------------------------------
# Predict official Kaggle test data
# ---------------------------------------------------------

final_xgb_test_prob = (
    final_xgb_model.predict_proba(
        X_test_ready
    )[:, 1]
)

print(
    "Number of test predictions:",
    len(final_xgb_test_prob)
)

print(
    "Minimum probability:",
    final_xgb_test_prob.min()
)

print(
    "Maximum probability:",
    final_xgb_test_prob.max()
)

print(
    "Mean probability:",
    final_xgb_test_prob.mean()
)

assert len(final_xgb_test_prob) == 48744
assert np.isfinite(final_xgb_test_prob).all()

assert (
    (final_xgb_test_prob >= 0)
    &
    (final_xgb_test_prob <= 1)
).all()

Number of test predictions: 48744
Minimum probability: 0.0014904914
Maximum probability: 0.71616626
Mean probability: 0.078919105


### 20.5 XGBoost Submission

The final full-data XGBoost model is used to generate an additional
Kaggle submission. This file is kept separate from the LightGBM
submission so that the two candidate models can be compared independently.

In [26]:
# ---------------------------------------------------------
# Create and validate XGBoost submission
# ---------------------------------------------------------

from pathlib import Path
import sys

sys.path.insert(0, str(ROOT / "src"))
from validate_submission import validate

submission_dir = ROOT / "results" / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

xgb_submission_path = (
    submission_dir / "submission_v2_xgboost.csv"
)

if xgb_submission_path.exists():
    raise FileExistsError(
        f"{xgb_submission_path} already exists. "
        "Compare the existing file before replacing it."
    )

xgb_submission = pd.DataFrame({
    "SK_ID_CURR": test["SK_ID_CURR"].to_numpy(),
    "TARGET": final_xgb_test_prob
})

sample_submission = pd.read_csv(
    ROOT / "data" / "raw" / "sample_submission.csv"
)

# Integrity checks
assert xgb_submission.shape == (48744, 2)

assert xgb_submission.columns.tolist() == [
    "SK_ID_CURR",
    "TARGET"
]

assert xgb_submission["SK_ID_CURR"].is_unique

assert (
    xgb_submission["SK_ID_CURR"].to_numpy()
    ==
    sample_submission["SK_ID_CURR"].to_numpy()
).all()

assert np.isfinite(
    xgb_submission["TARGET"]
).all()

assert xgb_submission["TARGET"].between(
    0, 1
).all()

# Save
xgb_submission.to_csv(
    xgb_submission_path,
    index=False
)

print("Saved to:")
print(xgb_submission_path)

print("\nSubmission shape:")
print(xgb_submission.shape)

print("\nFirst five rows:")
display(xgb_submission.head())

print("\nValidation:")
print(
    validate(
        xgb_submission_path,
        ROOT / "data/raw/sample_submission.csv",
        ROOT / "data/raw/application_test.csv"
    )
)

Saved to:
e:\MSDM5054\msdm5054-home-credit\results\submissions\submission_v2_xgboost.csv

Submission shape:
(48744, 2)

First five rows:


,SK_ID_CURR,TARGET
0,100001,0.022285
1,100005,0.126470
2,100013,0.011070
3,100028,0.039760
4,100038,0.163252



Validation:
PASS: 48744 rows, unique matching IDs, correct columns, valid probabilities; nothing submitted.


### 20.6 XGBoost Results and Updated Model Comparison

The XGBoost experiment is added to the C-part handoff outputs.

Because XGBoost achieved a slightly higher five-fold mean ROC-AUC and lower
cross-validation variability than the regularized LightGBM model, it is
selected as the final model for the current V1 modeling pipeline.

The previous LightGBM artifacts are retained for reproducibility and comparison.

In [27]:
# ---------------------------------------------------------
# Export XGBoost results and update C-part handoff
# ---------------------------------------------------------

import json
import joblib

C_RESULTS = ROOT / "results" / "c"
C_RESULTS.mkdir(parents=True, exist_ok=True)


# =========================================================
# 1. Updated overall model summary
# =========================================================

model_summary_updated = pd.DataFrame({
    "model": [
        "V0 Logistic Regression",
        "V0 LightGBM",
        "V1 LightGBM baseline",
        "V1 LightGBM baseline",
        "V1 Regularized LightGBM",
        "V1 Regularized LightGBM + class weight",
        "V1 Regularized LightGBM",
        "V1 XGBoost",
        "V1 XGBoost"
    ],

    "evaluation": [
        "Fixed 80/20",
        "Fixed 80/20",
        "Fixed 80/20",
        "5-fold CV mean",
        "5-fold CV mean",
        "5-fold CV mean",
        "Fixed 80/20",
        "Fixed 80/20",
        "5-fold CV mean"
    ],

    "roc_auc": [
        0.751239,
        0.759364,
        0.769522,
        0.763755,
        0.766092,
        0.765550,
        0.772396,
        xgb_fixed_auc,
        xgb_cv_mean_auc
    ]
})

model_summary_updated.to_csv(
    C_RESULTS / "model_summary.csv",
    index=False
)

display(model_summary_updated)


# =========================================================
# 2. XGBoost fold-level CV results
# =========================================================

xgb_cv_results = pd.DataFrame({
    "fold": np.arange(1, 6),
    "xgboost_auc": xgb_fold_aucs,
    "best_boosting_rounds": xgb_best_iterations
})

xgb_cv_results.to_csv(
    C_RESULTS / "xgboost_cv_results.csv",
    index=False
)


# =========================================================
# 3. XGBoost fixed-validation predictions
# =========================================================

xgb_validation_predictions = pd.DataFrame({
    "SK_ID_CURR": train.iloc[val_idx]["SK_ID_CURR"].to_numpy(),
    "TARGET": y_v1_val.to_numpy(),
    "PRED_PROB": xgb_fixed_val_prob
})

xgb_validation_predictions.to_csv(
    C_RESULTS / "validation_predictions_xgboost.csv",
    index=False
)

assert xgb_validation_predictions.shape[0] == 61503
assert xgb_validation_predictions["SK_ID_CURR"].is_unique
assert np.isfinite(
    xgb_validation_predictions["PRED_PROB"]
).all()


# =========================================================
# 4. XGBoost model configuration
# =========================================================

xgb_config = {
    "model": "XGBoost",
    "feature_version": "V1",
    "feature_count": 83,

    "parameters": {
        "objective": "binary:logistic",
        "n_estimators": int(FINAL_XGB_N_ESTIMATORS),
        "learning_rate": 0.03,
        "max_depth": 6,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "reg_alpha": 0.1,
        "reg_lambda": 1.0,
        "tree_method": "hist",
        "eval_metric": "auc",
        "random_state": 42
    },

    "fixed_validation_auc": float(xgb_fixed_auc),
    "cv_mean_auc": float(xgb_cv_mean_auc),
    "cv_std_auc": float(xgb_cv_std_auc),
    "oof_auc": float(xgb_oof_auc),

    "cv_best_boosting_rounds": [
        int(x) for x in xgb_best_iterations
    ],

    "final_n_estimators": int(
        FINAL_XGB_N_ESTIMATORS
    ),

    "selected_as_final_model": True
}

with open(
    C_RESULTS / "final_xgboost_config.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        xgb_config,
        f,
        indent=2
    )


# =========================================================
# 5. XGBoost feature importance
# =========================================================

xgb_feature_importance = pd.DataFrame({
    "feature": X_full_ready.columns,
    "importance": final_xgb_model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

xgb_feature_importance.to_csv(
    C_RESULTS / "xgboost_feature_importance.csv",
    index=False
)


# =========================================================
# 6. Save final XGBoost model
# =========================================================

xgb_model_path = (
    C_RESULTS /
    "final_xgboost_preprocessed_input.joblib"
)

joblib.dump(
    final_xgb_model,
    xgb_model_path
)


# =========================================================
# 7. Selected-model record
# =========================================================

selected_model = {
    "selected_model": "V1 XGBoost",
    "selection_basis": (
        "Higher five-fold mean ROC-AUC and lower "
        "cross-validation standard deviation than "
        "regularized LightGBM."
    ),

    "xgboost_cv_mean_auc": float(
        xgb_cv_mean_auc
    ),

    "xgboost_cv_std_auc": float(
        xgb_cv_std_auc
    ),

    "lightgbm_cv_mean_auc": 0.766092,
    "lightgbm_cv_std_auc": 0.002189,

    "xgboost_fixed_auc": float(
        xgb_fixed_auc
    ),

    "lightgbm_fixed_auc": 0.772396
}

with open(
    C_RESULTS / "selected_model.json",
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        selected_model,
        f,
        indent=2
    )


# =========================================================
# Final check
# =========================================================

print("\nNew / updated C-part outputs:")

for path in [
    C_RESULTS / "model_summary.csv",
    C_RESULTS / "xgboost_cv_results.csv",
    C_RESULTS / "validation_predictions_xgboost.csv",
    C_RESULTS / "final_xgboost_config.json",
    C_RESULTS / "xgboost_feature_importance.csv",
    C_RESULTS / "final_xgboost_preprocessed_input.joblib",
    C_RESULTS / "selected_model.json"
]:
    print("-", path.relative_to(ROOT))

,model,evaluation,roc_auc
0,V0 Logistic Regression,Fixed 80/20,0.751239
1,V0 LightGBM,Fixed 80/20,0.759364
2,V1 LightGBM baseline,Fixed 80/20,0.769522
3,V1 LightGBM baseline,5-fold CV mean,0.763755
4,V1 Regularized LightGBM,5-fold CV mean,0.766092
5,V1 Regularized LightGBM + class weight,5-fold CV mean,0.765550
6,V1 Regularized LightGBM,Fixed 80/20,0.772396
7,V1 XGBoost,Fixed 80/20,0.772394
8,V1 XGBoost,5-fold CV mean,0.767210



New / updated C-part outputs:
- results\c\model_summary.csv
- results\c\xgboost_cv_results.csv
- results\c\validation_predictions_xgboost.csv
- results\c\final_xgboost_config.json
- results\c\xgboost_feature_importance.csv
- results\c\final_xgboost_preprocessed_input.joblib
- results\c\selected_model.json
